# 08 — LSTM + Multi-Head Attention: Classificador Temporal de Pré-Violência

## Objetivo
O YOLOv8 detecta objetos **por frame** — não tem memória do passado.  
Este notebook treina um **classificador temporal** que recebe uma sequência de frames detectados
e estima:
- `pre_violence_score` — probabilidade de escalada para violência (0–1)
- `violence_score` — probabilidade de violência ativa em curso (0–1)

### Arquitetura
```
Frame features (t=1..T)  →  input_proj  →  LSTM (2 camadas)  →  MultiHeadAttention (4 cabeças)
                                                                         ↓
                                                               MLP  →  Sigmoid
                                                                         ↓
                                               [pre_violence_score, violence_score]
```

### Features por frame
Extraídas pelo YOLOv8 treinado nos notebooks anteriores:
- `n_persons` — número de pessoas detectadas
- `n_violence` — bboxes de classe `violence`
- `n_pre_violence` — bboxes de classe `pre_violence`
- `max_violence_conf` — maior confiança de detecção de violência
- `max_pre_violence_conf` — maior confiança de pré-violência
- `mean_box_area` — área média das bboxes normalizadas
- `iou_density` — densidade de sobreposição entre bboxes (proxy de contato físico)
- `motion_delta` — variação de movimento entre frames consecutivos

### Modelos utilizados
- **Experiment A** (cliente only): `models/client_only_best.pt`
- **Experiment B** (combinado): `models/combined_best.pt`
- Salva: `models/temporal_lstm.pt`

In [ ]:
!pip install -q ultralytics opencv-python-headless torch torchvision matplotlib seaborn pandas tqdm scikit-learn

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

import cv2
from ultralytics import YOLO

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Paths
ROOT = Path("..")  # notebook is inside notebooks/
MODELS_DIR = ROOT / "models"
DATA_DIR   = ROOT / "data"
CLIENT_LABELED = DATA_DIR / "client_labeled"
TEMPORAL_DIR   = DATA_DIR / "temporal_sequences"  # will be created here

MODELS_DIR.mkdir(exist_ok=True)
TEMPORAL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1. Seleção do modelo base YOLO

In [ ]:
# Choose which YOLO weights to use as the frame-level detector
# Prefer combined model if available, fall back to client-only
if (MODELS_DIR / "combined_best.pt").exists():
    YOLO_WEIGHTS = MODELS_DIR / "combined_best.pt"
    print("Using combined model (Experiment B)")
elif (MODELS_DIR / "client_only_best.pt").exists():
    YOLO_WEIGHTS = MODELS_DIR / "client_only_best.pt"
    print("Using client-only model (Experiment A)")
elif (MODELS_DIR / "public_only_best.pt").exists():
    YOLO_WEIGHTS = MODELS_DIR / "public_only_best.pt"
    print("Using public-pretrained model (05a)")
else:
    raise FileNotFoundError(
        "No trained YOLO weights found in models/. "
        "Run notebooks 05a, 05b and/or 06 first."
    )

yolo = YOLO(str(YOLO_WEIGHTS))
print(f"YOLO model: {YOLO_WEIGHTS.name}")
print(f"Classes: {yolo.names}")

## 2. Extração de features temporais

Para cada vídeo labelado do cliente extraímos uma sequência de feature-vetores (um por frame),
que serão as entradas do LSTM.

In [ ]:
# ── Class indices from trained YOLO ──────────────────────────────────────────
CLASS_NAMES = yolo.names  # dict {0: 'person', 1: 'violence', 2: 'pre_violence'}
IDX_PERSON       = next((k for k, v in CLASS_NAMES.items() if "person"       in v.lower()), None)
IDX_VIOLENCE     = next((k for k, v in CLASS_NAMES.items() if "violence"     in v.lower() and "pre" not in v.lower()), None)
IDX_PRE_VIOLENCE = next((k for k, v in CLASS_NAMES.items() if "pre_violence" in v.lower() or "pre-violence" in v.lower()), None)

print(f"person={IDX_PERSON}, violence={IDX_VIOLENCE}, pre_violence={IDX_PRE_VIOLENCE}")

In [ ]:
def compute_iou_density(boxes_xyxyn):
    """Mean pairwise IoU across all detected person boxes — proxy for physical proximity."""
    n = len(boxes_xyxyn)
    if n < 2:
        return 0.0
    ious = []
    for i in range(n):
        for j in range(i + 1, n):
            b1 = boxes_xyxyn[i]
            b2 = boxes_xyxyn[j]
            ix1 = max(b1[0], b2[0]); iy1 = max(b1[1], b2[1])
            ix2 = min(b1[2], b2[2]); iy2 = min(b1[3], b2[3])
            inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
            a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
            a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
            union = a1 + a2 - inter
            ious.append(inter / union if union > 0 else 0.0)
    return float(np.mean(ious))


def extract_frame_features(result, prev_boxes_xyxyn=None):
    """Extract 8-dim feature vector from a single YOLO result."""
    boxes  = result.boxes
    cls    = boxes.cls.cpu().numpy() if boxes is not None and len(boxes) else np.array([])
    conf   = boxes.conf.cpu().numpy() if boxes is not None and len(boxes) else np.array([])
    xyxyn  = boxes.xyxyn.cpu().numpy() if boxes is not None and len(boxes) else np.zeros((0, 4))

    mask_v  = cls == IDX_VIOLENCE
    mask_pv = cls == IDX_PRE_VIOLENCE
    mask_p  = cls == IDX_PERSON

    n_persons       = int(mask_p.sum())
    n_violence      = int(mask_v.sum())
    n_pre_violence  = int(mask_pv.sum())
    max_v_conf      = float(conf[mask_v].max())  if mask_v.any()  else 0.0
    max_pv_conf     = float(conf[mask_pv].max()) if mask_pv.any() else 0.0

    areas = [(b[2] - b[0]) * (b[3] - b[1]) for b in xyxyn]
    mean_area = float(np.mean(areas)) if areas else 0.0

    person_boxes = xyxyn[mask_p].tolist() if mask_p.any() else []
    iou_density  = compute_iou_density(person_boxes)

    # Motion delta: mean displacement of box centroids vs. previous frame
    if prev_boxes_xyxyn is not None and len(prev_boxes_xyxyn) > 0 and len(xyxyn) > 0:
        cur_cx  = (xyxyn[:, 0] + xyxyn[:, 2]) / 2
        cur_cy  = (xyxyn[:, 1] + xyxyn[:, 3]) / 2
        prv_cx  = (prev_boxes_xyxyn[:, 0] + prev_boxes_xyxyn[:, 2]) / 2
        prv_cy  = (prev_boxes_xyxyn[:, 1] + prev_boxes_xyxyn[:, 3]) / 2
        n_match = min(len(cur_cx), len(prv_cx))
        motion_delta = float(np.mean(np.sqrt(
            (cur_cx[:n_match] - prv_cx[:n_match])**2 +
            (cur_cy[:n_match] - prv_cy[:n_match])**2
        )))
    else:
        motion_delta = 0.0

    return np.array([
        n_persons, n_violence, n_pre_violence,
        max_v_conf, max_pv_conf,
        mean_area, iou_density, motion_delta
    ], dtype=np.float32), xyxyn


FEATURE_NAMES = [
    "n_persons", "n_violence", "n_pre_violence",
    "max_violence_conf", "max_pre_violence_conf",
    "mean_box_area", "iou_density", "motion_delta"
]
INPUT_DIM = len(FEATURE_NAMES)
print(f"Feature vector dimension: {INPUT_DIM}")
print("Features:", FEATURE_NAMES)

In [ ]:
SEQ_LEN    = 32   # frames per sequence window
FRAME_STEP = 2    # sample every N frames (controls temporal resolution vs. speed)

def video_to_sequences(video_path, label, seq_len=SEQ_LEN, stride=None):
    """
    Run YOLO on a video and slice the resulting feature time-series
    into overlapping windows of length `seq_len`.

    Returns list of (sequence_array [seq_len, INPUT_DIM], label) tuples.
    """
    if stride is None:
        stride = seq_len // 2  # 50% overlap

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []

    all_features = []
    prev_boxes   = None
    frame_idx    = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % FRAME_STEP == 0:
            results = yolo(frame, verbose=False)
            feat, prev_boxes = extract_frame_features(results[0], prev_boxes)
            all_features.append(feat)
        frame_idx += 1

    cap.release()

    if len(all_features) < seq_len:
        # Pad short videos with zeros
        pad = seq_len - len(all_features)
        all_features += [np.zeros(INPUT_DIM, dtype=np.float32)] * pad

    sequences = []
    arr = np.stack(all_features)  # [T, INPUT_DIM]
    for start in range(0, len(arr) - seq_len + 1, stride):
        seq = arr[start : start + seq_len]
        sequences.append((seq, label))

    return sequences

print(f"Window: {SEQ_LEN} frames | Step: {FRAME_STEP} | Stride: {SEQ_LEN//2}")

In [ ]:
# ── Build dataset ─────────────────────────────────────────────────────────────
# Label mapping:
#   non_violence  →  [0, 0]  (pre_violence=0, violence=0)
#   violence      →  [0, 1]  (pre_violence=0, violence=1)
# NOTE: pre_violence clips (if present) → [1, 0]
#       For now client_labeled only has violence/ and non_violence/

LABEL_MAP = {
    "non_violence":  np.array([0.0, 0.0], dtype=np.float32),
    "violence":      np.array([0.0, 1.0], dtype=np.float32),
    "pre_violence":  np.array([1.0, 0.0], dtype=np.float32),
}

CACHE_FILE = TEMPORAL_DIR / "sequences_cache.npz"

def build_sequence_dataset(force_rebuild=False):
    if CACHE_FILE.exists() and not force_rebuild:
        print(f"Loading cached sequences from {CACHE_FILE}")
        data = np.load(CACHE_FILE, allow_pickle=True)
        return data["X"], data["y"]

    all_X, all_y = [], []

    for folder_name, label_vec in LABEL_MAP.items():
        folder = CLIENT_LABELED / folder_name
        if not folder.exists():
            print(f"  Skipping {folder_name} (not found)")
            continue

        videos = list(folder.glob("*.mp4")) + list(folder.glob("*.avi"))
        print(f"  {folder_name}: {len(videos)} vídeos")

        for vp in tqdm(videos, desc=folder_name, leave=False):
            seqs = video_to_sequences(vp, label_vec)
            for seq, lbl in seqs:
                all_X.append(seq)
                all_y.append(lbl)

    X = np.stack(all_X)  # [N, SEQ_LEN, INPUT_DIM]
    y = np.stack(all_y)  # [N, 2]

    np.savez_compressed(CACHE_FILE, X=X, y=y)
    print(f"Saved cache: {CACHE_FILE}")
    return X, y


X, y = build_sequence_dataset(force_rebuild=False)
print(f"\nTotal sequences: {len(X)}")
print(f"X shape: {X.shape}  |  y shape: {y.shape}")
print(f"Violence   (y[:,1]=1): {int((y[:,1]==1).sum())}")
print(f"No action  (y[:,0]=0, y[:,1]=0): {int(((y[:,0]==0)&(y[:,1]==0)).sum())}")

## 3. Dataset PyTorch e splits

In [ ]:
from sklearn.model_selection import train_test_split

# Single label for stratification: 0=neutral, 1=pre_violence, 2=violence
y_cls = np.argmax(np.hstack([1 - y.sum(axis=1, keepdims=True), y[:, [0]], y[:, [1]]]), axis=1)

idx_train, idx_tmp = train_test_split(np.arange(len(X)), test_size=0.3, stratify=y_cls, random_state=SEED)
idx_val,   idx_test = train_test_split(idx_tmp, test_size=0.5, stratify=y_cls[idx_tmp], random_state=SEED)

print(f"Train: {len(idx_train)} | Val: {len(idx_val)} | Test: {len(idx_test)}")


class TemporalDataset(Dataset):
    def __init__(self, X, y, indices):
        self.X = torch.tensor(X[indices], dtype=torch.float32)
        self.y = torch.tensor(y[indices], dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


BATCH_SIZE = 32
ds_train = TemporalDataset(X, y, idx_train)
ds_val   = TemporalDataset(X, y, idx_val)
ds_test  = TemporalDataset(X, y, idx_test)

dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True)
dl_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE)
dl_test  = DataLoader(ds_test,  batch_size=BATCH_SIZE)

print(f"Batches — train: {len(dl_train)} | val: {len(dl_val)} | test: {len(dl_test)}")

## 4. Modelo: LSTM + Multi-Head Attention

In [ ]:
class ViolenceLSTM(nn.Module):
    """
    input_proj  : Linear(INPUT_DIM → hidden_dim)
    lstm        : LSTM(hidden_dim, hidden_dim, num_layers=2, bidirectional=True)
    attention   : MultiheadAttention(2*hidden_dim, num_heads=4)
    mlp         : Linear(2*hidden_dim → output_dim=2)
    activation  : Sigmoid  (multi-label: both scores independent)
    """

    def __init__(self, input_dim=INPUT_DIM, hidden_dim=64, num_layers=2,
                 num_heads=4, dropout=0.3, output_dim=2):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        lstm_out_dim = hidden_dim * 2  # bidirectional
        self.attn = nn.MultiheadAttention(
            embed_dim=lstm_out_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.attn_norm = nn.LayerNorm(lstm_out_dim)

        self.mlp = nn.Sequential(
            nn.Linear(lstm_out_dim, lstm_out_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_out_dim // 2, output_dim),
        )

    def forward(self, x):
        # x: [B, T, input_dim]
        x = self.input_proj(x)           # [B, T, hidden_dim]
        lstm_out, _ = self.lstm(x)       # [B, T, 2*hidden_dim]

        # Self-attention over time
        attn_out, _ = self.attn(lstm_out, lstm_out, lstm_out)
        attn_out = self.attn_norm(lstm_out + attn_out)  # residual

        # Global average pool across time
        pooled = attn_out.mean(dim=1)    # [B, 2*hidden_dim]

        logits = self.mlp(pooled)        # [B, 2]
        return torch.sigmoid(logits)     # scores in [0, 1]


model = ViolenceLSTM().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nParâmetros treináveis: {total_params:,}")

## 5. Treinamento

In [ ]:
EPOCHS    = 50
LR        = 3e-4
PATIENCE  = 10    # early stopping

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.BCELoss()

history = {"train_loss": [], "val_loss": [], "val_auc": []}
best_val_loss = float("inf")
patience_ctr  = 0
CKPT_PATH = MODELS_DIR / "temporal_lstm.pt"

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    train_losses = []
    for xb, yb in dl_train:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item())

    scheduler.step()

    # ── Validate ──
    model.eval()
    val_losses, all_pred, all_true = [], [], []
    with torch.no_grad():
        for xb, yb in dl_val:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb)
            val_losses.append(criterion(pred, yb).item())
            all_pred.append(pred.cpu().numpy())
            all_true.append(yb.cpu().numpy())

    all_pred = np.vstack(all_pred)
    all_true = np.vstack(all_true)

    tl = np.mean(train_losses)
    vl = np.mean(val_losses)

    # AUC per output
    try:
        auc_pv = roc_auc_score(all_true[:, 0], all_pred[:, 0])
    except ValueError:
        auc_pv = float("nan")
    try:
        auc_v  = roc_auc_score(all_true[:, 1], all_pred[:, 1])
    except ValueError:
        auc_v = float("nan")

    history["train_loss"].append(tl)
    history["val_loss"].append(vl)
    history["val_auc"].append((auc_pv, auc_v))

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | Train: {tl:.4f} | Val: {vl:.4f} | AUC pv={auc_pv:.3f} v={auc_v:.3f}")

    # Early stopping
    if vl < best_val_loss:
        best_val_loss = vl
        patience_ctr  = 0
        torch.save(model.state_dict(), CKPT_PATH)
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nBest val loss: {best_val_loss:.4f}")
print(f"Model saved to: {CKPT_PATH}")

## 6. Curvas de treinamento

In [ ]:
epochs_ran = len(history["train_loss"])
ep_axis    = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
axes[0].plot(ep_axis, history["train_loss"], label="Train Loss")
axes[0].plot(ep_axis, history["val_loss"],   label="Val Loss")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

# AUC
auc_pv_list = [a[0] for a in history["val_auc"]]
auc_v_list  = [a[1] for a in history["val_auc"]]
axes[1].plot(ep_axis, auc_pv_list, label="AUC pre_violence")
axes[1].plot(ep_axis, auc_v_list,  label="AUC violence")
axes[1].set_title("Validation AUC")
axes[1].set_xlabel("Epoch")
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("LSTM + Multi-Head Attention — Treinamento", fontsize=13)
plt.tight_layout()
plt.savefig(MODELS_DIR / "lstm_training_curves.png", dpi=150)
plt.show()
print("Saved: models/lstm_training_curves.png")

## 7. Avaliação no conjunto de teste

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

all_pred_test, all_true_test = [], []
with torch.no_grad():
    for xb, yb in dl_test:
        pred = model(xb.to(DEVICE))
        all_pred_test.append(pred.cpu().numpy())
        all_true_test.append(yb.numpy())

pred_test = np.vstack(all_pred_test)
true_test = np.vstack(all_true_test)

THRESHOLD = 0.5
pred_bin = (pred_test >= THRESHOLD).astype(int)

print("=== pre_violence ===")
print(classification_report(true_test[:, 0].astype(int), pred_bin[:, 0],
                             target_names=["sem pre-viol", "pre-viol"], zero_division=0))

print("=== violence ===")
print(classification_report(true_test[:, 1].astype(int), pred_bin[:, 1],
                             target_names=["sem violência", "violência"], zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for i, (cls_name, ax) in enumerate(zip(["pre_violence", "violence"], axes)):
    cm = confusion_matrix(true_test[:, i].astype(int), pred_bin[:, i])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Pred 0", "Pred 1"],
                yticklabels=["True 0", "True 1"])
    ax.set_title(f"Confusion Matrix — {cls_name}")

plt.tight_layout()
plt.savefig(MODELS_DIR / "lstm_confusion_matrices.png", dpi=150)
plt.show()
print("Saved: models/lstm_confusion_matrices.png")

In [ ]:
# AUC scores
metrics = {}
for i, cls_name in enumerate(["pre_violence", "violence"]):
    try:
        auc = roc_auc_score(true_test[:, i], pred_test[:, i])
    except ValueError:
        auc = None
    metrics[cls_name] = {"auc_roc": auc, "threshold": THRESHOLD}
    print(f"{cls_name}: AUC-ROC = {auc:.4f}" if auc is not None else f"{cls_name}: AUC-ROC = N/A (single class)")

# Save metrics
metrics_path = MODELS_DIR / "results_lstm.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nMetrics saved to {metrics_path}")

## 8. Inferência rápida em um vídeo de exemplo

In [ ]:
def predict_video(video_path, seq_len=SEQ_LEN):
    """
    Run the temporal classifier on a full video.
    Returns a DataFrame with columns: [frame_start, pre_violence_score, violence_score]
    """
    model.eval()
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    all_features = []
    prev_boxes   = None
    fi = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if fi % FRAME_STEP == 0:
            results = yolo(frame, verbose=False)
            feat, prev_boxes = extract_frame_features(results[0], prev_boxes)
            all_features.append(feat)
        fi += 1
    cap.release()

    if len(all_features) < seq_len:
        pad = seq_len - len(all_features)
        all_features += [np.zeros(INPUT_DIM, dtype=np.float32)] * pad

    arr = np.stack(all_features)
    stride = seq_len // 2
    records = []

    with torch.no_grad():
        for start in range(0, len(arr) - seq_len + 1, stride):
            window = arr[start : start + seq_len]
            x = torch.tensor(window).unsqueeze(0).to(DEVICE)  # [1, T, D]
            scores = model(x)[0].cpu().numpy()
            records.append({
                "frame_start": start * FRAME_STEP,
                "pre_violence_score": float(scores[0]),
                "violence_score":     float(scores[1]),
            })

    return pd.DataFrame(records)


# Find any test video to demo
demo_video = None
for folder in [CLIENT_LABELED / "violence", CLIENT_LABELED / "non_violence",
               DATA_DIR / "client_videos"]:
    candidates = list(folder.glob("*.mp4")) + list(folder.glob("*.avi")) if folder.exists() else []
    if candidates:
        demo_video = candidates[0]
        break

if demo_video:
    print(f"Demo video: {demo_video.name}")
    df_scores = predict_video(demo_video)
    print(df_scores.head(10))
else:
    print("No demo video found. Run notebook 02 first to label client clips.")
    df_scores = None

In [ ]:
if df_scores is not None and not df_scores.empty:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(df_scores["frame_start"], df_scores["pre_violence_score"],
            label="pre_violence", color="orange", linewidth=1.5)
    ax.plot(df_scores["frame_start"], df_scores["violence_score"],
            label="violence", color="red", linewidth=1.5)
    ax.axhline(THRESHOLD, color="gray", linestyle="--", label=f"threshold={THRESHOLD}")
    ax.fill_between(df_scores["frame_start"], THRESHOLD, df_scores["violence_score"],
                    where=df_scores["violence_score"] >= THRESHOLD,
                    alpha=0.25, color="red", label="violência detectada")
    ax.set_xlabel("Frame")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1)
    ax.legend()
    ax.set_title(f"Scores temporais — {demo_video.name}")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 9. Resumo

| Arquivo | Descrição |
|---|---|
| `models/temporal_lstm.pt` | Pesos do melhor modelo LSTM |
| `models/lstm_training_curves.png` | Curvas de loss e AUC |
| `models/lstm_confusion_matrices.png` | Matrizes de confusão por classe |
| `models/results_lstm.json` | Métricas AUC por classe |
| `data/temporal_sequences/sequences_cache.npz` | Cache de features (evita reprocessar) |

**Próximo passo:** `09_inference_demo.ipynb` — demo completo com RTSP e processamento de arquivos.